# 2 · Building and running a graph

A graph is a record: its nodes, each naming the node it uses and what is typed into it, and its edges. An input takes an edge, a value (fixed), a default (open to a caller), or nothing (its declared default applies). `registry.load` reads the record into a `Graph` and never raises for a fault in it. `graph.run()` runs it and returns the `Run` it ended on; `run.stream()` says what happens as it happens.

In [ ]:
from typing import Annotated

from conductor import GraphRecord, NodeRecord, NodeRegistry, Param, Series
from conductor.widgets import TextInput

registry = NodeRegistry(category="text")


@registry.node("echo", title="Echo", description="Returns its input")
def echo(text: Annotated[str, Param(title="Input", widget=TextInput())]) -> Annotated[str, Param(title="Output")]:
    return text


@registry.node("upper", title="Uppercase", description="Uppercases a text")
def upper(text: Annotated[str, Param(title="Input", widget=TextInput())]) -> Annotated[str, Param(title="Output")]:
    return text.upper()


@registry.node("pair", title="Pair", description="Joins two texts")
def pair(
    a: Annotated[str, Param(title="A", widget=TextInput())],
    b: Annotated[str, Param(title="B", widget=TextInput())],
    separator: Annotated[str, Param(title="Separator", widget=TextInput())] = " + ",
) -> Annotated[str, Param(title="Result")]:
    return f"{a}{separator}{b}"


@registry.node("join-all", title="Join all", description="Joins every text it is given")
def join_all(texts: Annotated[Series[str], Param(title="Texts")]) -> Annotated[str, Param(title="Result")]:
    return " ".join(texts)

## Build the graph

```text
  hello ──> shout ──┐
                    ├──> pair
  world ────────────┘
```

In [ ]:
record = GraphRecord(
    nodes=[
        NodeRecord(id="hello", node="echo", defaults={"text": "hello"}),
        NodeRecord(id="world", node="echo", defaults={"text": "world"}),
        NodeRecord(id="shout", node="upper"),
        NodeRecord(id="pair", node="pair", values={"separator": " & "}),
    ],
    edges=[("hello.result", "shout.text"), ("shout.result", "pair.a"), ("world.result", "pair.b")],
)

graph = registry.load(record)
print("runnable:", graph.is_runnable)
print("plan:    ", graph.plan)

## Ask the loaded graph

`graph.plan` says where each node runs, `graph.inputs` and `graph.outputs` what it takes and gives, and `graph.describe()` is all of it as JSON, each node's fields included.

In [ ]:
pair_inputs = graph.describe()["nodes"]["pair"]["inputs"]
print("pair takes:", [(field["name"], field["type"]) for field in pair_inputs])

## Problems, not exceptions

A broken graph loads. Each problem has a stable `type` and a place, and running a graph with one raises `InvalidCallError`.

In [ ]:
broken = registry.load(GraphRecord(nodes=[NodeRecord(id="shout", node="upper")], edges=[("ghost.result", "shout.text")]))
print("runnable:", broken.is_runnable)
for problem in broken.problems:
    print(f"  {problem.type} on {problem.loc[0]}: {problem.msg}")

## Run it

`run.results` is `{node_id: {output: value}}`.

In [ ]:
run = graph.run()
for node_id, outputs in run.results.items():
    print(f"  {node_id}: {outputs['result']!r}")

## Fill its inputs, read its outputs

A graph takes the open inputs of its first nodes, by bare name, or by address when two share a name. `run.outputs` is what the graph returns, by address.

In [ ]:
print("takes:  ", list(graph.inputs))
print("returns:", graph.outputs)

graph.run({"hello.text": "goodbye"}).outputs

## Watch the events

In [ ]:
async for event in graph.with_inputs().stream():
    match event.type:
        case "node_start":
            print(f"  start     {event.node_id}")
        case "node_complete":
            print(f"  complete  {event.node_id}: {event.rows} row(s)")
        case "run_end":
            print(f"  done      {event.run.status}")
        case other:
            print(f"  {other}")

## Once per row

An open input marked `many` takes a list from the caller, and a node typed for one value runs once per item. A `Series[X]` input takes every value in one call. `make_graph` writes a record from Python: with no `edges` the nodes form a chain, each node under its own id.

In [ ]:
rows = registry.load({
    "nodes": [{"id": "upper", "node": "upper", "many": ["text"]}, {"id": "join-all", "node": "join-all"}],
    "edges": [["upper.result", "join-all.texts"]],
})
print(rows.plan)

async for event in rows.with_inputs(text=["one", "two", "three"]).stream():
    if event.type == "node_progress":
        print(f"  {event.node_id}: {event.done} of {event.total}")
    elif event.type == "run_end":
        print("  upper:   ", event.run.results["upper"]["result"])
        print("  join-all:", event.run.results["join-all"]["result"])

## Save it

A `GraphRecord` reads back the JSON it wrote; empty fields are left out.

In [ ]:
text = record.model_dump_json()
print(text)
assert GraphRecord.model_validate_json(text) == record